In [1]:
import pandas as pd 

In [72]:
df = pd.read_csv('nashville_tennessee')
df.head(2)

,id,host_since,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,availability_30,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,reviews_per_month
0,6422,2009-04-03,within a day,1.0,1.00,False,1,True,District 6,36.17143,-86.73570,Private room in home,Private room,1,1.0,1.0,1.0,"[""Dryer \u2013 In building"", ""Kayak"", ""Carbon ...",43.0,30,365,0,175,667,0,0,0.0,4.95,4.96,4.92,False,3.34
1,39870,2010-07-18,within an hour,1.0,0.91,True,2,True,District 25,36.12466,-86.81269,Private room in home,Private room,2,1.0,1.0,1.0,"[""Carbon monoxide alarm"", ""Microwave"", ""Centra...",70.0,1,7,17,242,587,80,255,17850.0,4.93,4.91,4.93,False,5.34


#### 1. host_experience_years

In [73]:
df['host_since'] = pd.to_datetime(df['host_since'])
df['host_experience_years'] = ( (pd.Timestamp.today() - df['host_since']).dt.days / 365.25 ).round(2)
df.drop(columns='host_since', inplace=True)

In [50]:
df.head(2)

,id,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_total_listings_count,host_identity_verified,neighbourhood_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,availability_30,availability_365,number_of_reviews,number_of_reviews_ltm,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_cleanliness,review_scores_location,instant_bookable,reviews_per_month,host_experience_years
0,6422,within a day,1.0,1.00,False,1,True,District 6,36.17143,-86.73570,Private room in home,Private room,1,1.0,1.0,1.0,"[""Dryer \u2013 In building"", ""Kayak"", ""Carbon ...",43.0,30,365,0,175,667,0,0,0.0,4.95,4.96,4.92,False,3.34,17.25
1,39870,within an hour,1.0,0.91,True,2,True,District 25,36.12466,-86.81269,Private room in home,Private room,2,1.0,1.0,1.0,"[""Carbon monoxide alarm"", ""Microwave"", ""Centra...",70.0,1,7,17,242,587,80,255,17850.0,4.93,4.91,4.93,False,5.34,15.96


#### 2. Simplifying property_type

In [74]:
# This bundles low-to-mid frequency categories into distinct pricing tiers
custom_buckets = {
    # Boutique Hospitality (higher price tier)
    "Room in hotel": "Hospitality_Boutique",
    "Room in boutique hotel": "Hospitality_Boutique",
    "Room in aparthotel": "Hospitality_Boutique",
    "Entire serviced apartment": "Hospitality_Boutique",
    "Casa particular": "Hospitality_Boutique",
    # Luxury & Unique Spaces (premium price tier)
    "Entire villa": "Luxury & Unique",
    "Entire resort": "Luxury & Unique",
    "Boat": "Luxury & Unique",
    "Treehouse": "Luxury & Unique",
    "Dome": "Luxury & Unique",
    "Entire chalet": "Luxury & Unique",
    # Mid-tier Standalone Spaces
    "Entire bungalow": "Guesthouse_Bungalow",
    "Entire cottage": "Guesthouse_Bungalow",
    "Entire cabin": "Guesthouse_Bungalow",
    "Tiny home": "Guesthouse_Bungalow",
    "Barn": "Guesthouse_Bungalow",
    "Farm stay": "Guesthouse_Bungalow",
    "Private room in farm stay": "Guesthouse_Bungalow",
    "Private room in nature lodge": "Guesthouse_Bungalow",
    # Guesthouses & Suites
    "Entire guest suite": "Guesthouses & Suites",
    "Entire guesthouse": "Guesthouses & Suites",
    "Private room in guest suite": "Guesthouses & Suites",
    "Private room in guesthouse": "Guesthouses & Suites",
    # Budget / Shared Rooms (lowest price tier)
    "Private room in hostel": "Budget & Shared",
    "Shared room in home": "Budget & Shared",
    "Tent": "Budget & Shared",
    "Camper/RV": "Budget & Shared",
    "Shipping container": "Budget & Shared",
}

df["engineered_property"] = df["property_type"].map(custom_buckets)

# If a category wasn't in the dictionary, it keeps its original name here
df["engineered_property"] = df["engineered_property"].fillna(df["property_type"])

# This pulls the loose, fragmented private room types into one highly accurate pricing signal
private_room_cleanup = {
    "Private room in home": "Private room",
    "Private room in townhouse": "Private room",
    "Private room in rental unit": "Private room",
    "Private room": "Private room",
    "Private room in bed and breakfast": "Private room",
}
df["engineered_property"] = df["engineered_property"].replace(
    private_room_cleanup
)

# This automatically catches any tiny remnants with fewer than 20 rows and bundles them
counts = df["engineered_property"].value_counts()
rare_categories = counts[counts < 20].index
df["engineered_property"] = df["engineered_property"].replace(
    rare_categories, "Rare"
)

df["engineered_property"].value_counts()

engineered_property
Entire home             2547
Entire rental unit      1396
Entire condo             729
Entire townhouse         709
Guesthouses & Suites     444
Private room             386
Hospitality_Boutique     145
Guesthouse_Bungalow      113
Entire loft               96
Rare                      43
Budget & Shared           21
Name: count, dtype: int64

In [75]:
df.drop(columns='property_type', inplace=True)

#### 3. amenity_count and Premium amenities

In [76]:
import ast

df['amenity_count'] = (
    df['amenities']
      .apply(lambda x: len(ast.literal_eval(x)))
)

amenities = df['amenities'].str.lower()

df['has_pool'] = amenities.str.contains('pool')
df['has_gym'] = amenities.str.contains('gym')
df['has_free_parking'] = amenities.str.contains('free parking')

df.drop(columns='amenities', inplace=True)

#### 4. Host portfolio size

In [59]:
bins = [0, 1, 10, float('inf')]

labels = [
    'single',
    'small',
    'professional'
]

df['host_portfolio_size'] = pd.cut(
    df['host_total_listings_count'],
    bins=bins,
    labels=labels
)

df.drop(columns='host_total_listings_count', inplace=True)

In [82]:
df = df[
    [
        'id',
        'engineered_property',
        'room_type',
        'neighbourhood_cleansed',
        'latitude',
        'longitude',

        'accommodates',
        'bedrooms',
        'beds',
        'bathrooms',

        'price',
        'estimated_revenue_l365d',
        'estimated_occupancy_l365d',

        'host_experience_years',
        'host_total_listings_count',
        'host_identity_verified',
        'host_is_superhost',
        'host_response_time',
        'host_response_rate',
        'host_acceptance_rate',
        'instant_bookable',

        'amenity_count',
        'has_pool',
        'has_gym',
        'has_free_parking',

        'number_of_reviews',
        'number_of_reviews_ltm',
        'reviews_per_month',
        'review_scores_rating',
        'review_scores_cleanliness',
        'review_scores_location',

        'minimum_nights',
        'maximum_nights',
        'availability_30',
        'availability_365'
    ]
]

In [84]:
df.rename(
    columns={
        "engineered_property": "property_category",
        "host_total_listings_count": "host_listing_count",
        "estimated_revenue_l365d":"estimated_annual_revenue",
        "estimated_occupancy_l365d":"estimated_annual_occupancy",
        "availability_365":"availability_year",
        "availability_30":"availability_monthly",
        "neighbourhood_cleansed":"neighbourhood",
        "number_of_reviews_ltm":"annual_reviews",
        "number_of_reviews_ltm":"monthly_review",
        "number_of_review":"total_reviews",
        "review_scores_rating":"rating",
        "review_scores_cleanliness":"cleanliness_rating",
        "review_scores_location":"location_rating",
    },
    inplace=True,
)

In [94]:
df = df[df['price'] != 0]

In [87]:
pd.set_option('display.max_columns', None)
df.head(5)

,id,property_category,room_type,neighbourhood,latitude,longitude,accommodates,bedrooms,beds,bathrooms,price,estimated_annual_revenue,estimated_annual_occupancy,host_experience_years,host_listing_count,host_identity_verified,host_is_superhost,host_response_time,host_response_rate,host_acceptance_rate,instant_bookable,amenity_count,has_pool,has_gym,has_free_parking,number_of_reviews,monthly_review,reviews_per_month,rating,cleanliness_rating,location_rating,minimum_nights,maximum_nights,availability_monthly,availability_year
0,6422,Private room,Private room,District 6,36.17143,-86.73570,1,1.0,1.0,1.0,43.0,0.0,0,17.25,1,True,False,within a day,1.0,1.00,False,52,False,False,False,667,0,3.34,4.95,4.96,4.92,30,365,0,175
1,39870,Private room,Private room,District 25,36.12466,-86.81269,2,1.0,1.0,1.0,70.0,17850.0,255,15.96,2,True,True,within an hour,1.0,0.91,False,27,False,False,True,587,80,5.34,4.93,4.91,4.93,1,7,17,242
2,72906,Entire rental unit,Entire home/apt,District 18,36.13122,-86.80066,2,2.0,2.0,1.0,91.0,18564.0,204,15.95,1,True,True,within an hour,1.0,0.96,False,42,False,False,False,777,34,4.46,4.92,4.83,4.97,2,7,8,234
3,258817,Private room,Private room,District 12,36.16076,-86.59151,2,1.0,2.0,1.0,34.0,0.0,0,17.03,25,True,False,within an hour,1.0,1.00,False,33,False,False,True,97,0,0.58,4.78,4.41,4.74,30,365,29,364
4,289242,Private room,Private room,District 12,36.16296,-86.59113,1,2.0,2.0,1.0,33.0,0.0,0,17.03,25,True,False,within an hour,1.0,1.00,False,31,False,False,True,76,0,0.45,4.71,4.27,4.47,30,365,30,365


In [96]:
df.to_csv('nashville_tennessee II', index=False)